In [15]:
import math


def wisric_moid(saxisA, eccenA, argpeA, omegaA, incliA, 
                saxisB, eccenB, argpeB, omegaB, incliB):

    # --- 1. Constants ---
    AU_TO_KM = 149597870.7
    twopi = 2 * math.pi
    cstep = 0.12        
    stepini = 0.07      
    steptresh = 1E-5    
    stepmin = 1E-14     

    # --- 2. Preparing the Orbits ---
    # Rotate Orbit B relative to Orbit A
    c11 = math.cos(omegaA)*math.cos(argpeA) - math.sin(omegaA)*math.cos(incliA)*math.sin(argpeA)
    c12 = math.sin(omegaA)*math.cos(argpeA) + math.cos(omegaA)*math.cos(incliA)*math.sin(argpeA)
    c13 = math.sin(incliA)*math.sin(argpeA)
    c21 = -math.cos(omegaA)*math.sin(argpeA) - math.sin(omegaA)*math.cos(incliA)*math.cos(argpeA)
    c22 = -math.sin(omegaA)*math.sin(argpeA) + math.cos(omegaA)*math.cos(incliA)*math.cos(argpeA)
    c23 = math.sin(incliA)*math.cos(argpeA)
    c31 = math.sin(incliA)*math.sin(omegaA)
    c32 = -math.sin(incliA)*math.cos(omegaA)
    c33 = math.cos(incliA)

    sintmpi = math.sin(incliB)
    costmpi = math.cos(incliB)
    costmpo = math.cos(omegaB)
    sintmpo = math.sin(omegaB)
    costmpa = math.cos(argpeB)
    sintmpa = math.sin(argpeB)

    x1 = costmpo*costmpa - sintmpo*costmpi*sintmpa
    x2 = sintmpo*costmpa + costmpo*costmpi*sintmpa
    x3 = sintmpi*sintmpa
    y1 = -costmpo*sintmpa - sintmpo*costmpi*costmpa
    y2 = -sintmpo*sintmpa + costmpo*costmpi*costmpa
    y3 = sintmpi*costmpa
    z1 = sintmpi*sintmpo
    z2 = -sintmpi*costmpo
    z3 = costmpi

    z1n = c11*z1 + c12*z2 + c13*z3
    z2n = c21*z1 + c22*z2 + c23*z3
    z3n = c31*z1 + c32*z2 + c33*z3
    y3n = c31*y1 + c32*y2 + c33*y3
    x3n = c31*x1 + c32*x2 + c33*x3

    incliB = math.atan2(math.sqrt(z1n*z1n + z2n*z2n), z3n)
    omegaB = -math.atan2(z1n, -z2n)
    argpeB = -math.atan2(x3n, y3n)

    costmpo = math.cos(omegaB)
    sintmpo = math.sin(omegaB)
    sintmpi = math.sin(incliB) 
    costmpi = z3n              
    sint = sintmpo * costmpi
    cost = costmpo * costmpi
    
    radA = saxisA * (1.0 - eccenA*eccenA)
    radB = saxisB * (1.0 - eccenB*eccenB)

    # --- 3. Scanning Phase ---
    tmptrueB = []
    tmplongit = []
    tmpmoid = []

    trueB = -2.0 * cstep
    dist_o = 1E6
    dist_oo = 0
    trueB_o = 0
    longit_o = 0
    dist_min = 1E6 

    # Prime the previous distance check
    for iii in range(1, 3):
        rB = radB / (1.0 + eccenB * math.cos(trueB))
        sintmp = math.sin(trueB + argpeB)
        costmp = math.cos(trueB + argpeB)
        Bz_sq = sintmpi * sintmp
        Bz_sq = Bz_sq * Bz_sq
        
        longit = math.atan2(sintmpo*costmp + sintmp*cost, costmpo*costmp - sintmp*sint)
        
        tmp2 = eccenA * math.cos(longit)
        rA = radA / (1.0 + tmp2)
        rA2 = radA / (1.0 - tmp2)
        tmp1 = rB * math.sqrt(max(0, 1.0 - Bz_sq)) 

        if abs(tmp1 - rA) > abs(tmp1 + rA2):
            rA = rA2
            longit = longit - math.pi
            tmp1 = tmp1 + rA2
        else:
            tmp1 = tmp1 - rA
            
        dist = rB*rB*Bz_sq + tmp1*tmp1
        
        if iii == 1:
            dist_oo = dist
        else:
            dist_o = dist
            trueB_o = trueB
            longit_o = longit
            
        trueB += cstep

    # Full revolution scan
    dist_min = dist
    while trueB < (twopi + cstep):
        rB = radB / (1.0 + eccenB * math.cos(trueB))
        sintmp = math.sin(trueB + argpeB)
        costmp = math.cos(trueB + argpeB)
        Bz_sq = sintmpi * sintmp
        Bz_sq = Bz_sq * Bz_sq

        longit = math.atan2(sintmpo*costmp + sintmp*cost, costmpo*costmp - sintmp*sint)
        
        tmp2 = eccenA * math.cos(longit)
        rA = radA / (1.0 + tmp2)
        rA2 = radA / (1.0 - tmp2)
        tmp1 = rB * math.sqrt(max(0, 1.0 - Bz_sq))

        if abs(tmp1 - rA) > abs(tmp1 + rA2):
            rA = rA2
            longit = longit - math.pi
            tmp1 = tmp1 + rA2
        else:
            tmp1 = tmp1 - rA

        dist = rB*rB*Bz_sq + tmp1*tmp1

        if (dist_o <= dist) and (dist_o <= dist_oo):
            tmptrueB.append(trueB_o)
            tmplongit.append(longit_o)
            tmpmoid.append(dist_o)

        if dist_min > dist:
            dist_min = dist

        dist_oo = dist_o
        dist_o = dist
        trueB_o = trueB
        longit_o = longit
        trueB += cstep

    # --- 4. Tuning Phase ---
    nmax = len(tmpmoid)
    final_moid = 1E6
    final_trueB = 0.0
    final_longit = 0.0
    
    # Internal tuning function
    def run_tuning(t_moid, t_trueB, t_longit, t_step, t_threshold):
        curr_moid = t_moid
        curr_trueB = t_trueB
        curr_longit = t_longit
        curr_step = t_step
        
        t_aleft, t_aright, t_bleft, t_bright = True, True, True, True
        
        # Grid arrays (1-based index emulation)
        _rBt = [0.0]*4; _Bxt = [0.0]*4; _Byt = [0.0]*4; _Bzt = [0.0]*4
        _rAt = [0.0]*4; _Axt = [0.0]*4; _Ayt = [0.0]*4
        
        # Center point setup
        t_rBt2 = radB / (1.0 + eccenB * math.cos(curr_trueB))
        s_tmp = math.sin(curr_trueB + argpeB)
        c_tmp = math.cos(curr_trueB + argpeB)
        t_Bxt2 = costmpo*c_tmp - s_tmp*sint
        t_Byt2 = sintmpo*c_tmp + s_tmp*cost
        t_Bzt2 = sintmpi*s_tmp
        
        t_rAt2 = radA / (1.0 + eccenA * math.cos(curr_longit))
        t_Axt2 = math.cos(curr_longit)
        t_Ayt2 = math.sin(curr_longit)
        
        _rBt[2]=t_rBt2; _Bxt[2]=t_Bxt2; _Byt[2]=t_Byt2; _Bzt[2]=t_Bzt2
        _rAt[2]=t_rAt2; _Axt[2]=t_Axt2; _Ayt[2]=t_Ayt2

        while curr_step >= t_threshold:
            lpoints = 0
            j1min, j1max = 1, 3
            i1min, i1max = 1, 3
            calc1, calc2, calc3, calc4 = False, False, False, False

            if t_bleft:
                _rBt[1] = radB / (1.0 + eccenB * math.cos(curr_trueB - curr_step))
                s_tmp = math.sin(curr_trueB - curr_step + argpeB)
                c_tmp = math.cos(curr_trueB - curr_step + argpeB)
                _Bxt[1] = costmpo*c_tmp - s_tmp*sint
                _Byt[1] = sintmpo*c_tmp + s_tmp*cost
                _Bzt[1] = sintmpi*s_tmp
                lpoints += 1
            
            if t_bright:
                _rBt[3] = radB / (1.0 + eccenB * math.cos(curr_trueB + curr_step))
                s_tmp = math.sin(curr_trueB + curr_step + argpeB)
                c_tmp = math.cos(curr_trueB + curr_step + argpeB)
                _Bxt[3] = costmpo*c_tmp - s_tmp*sint
                _Byt[3] = sintmpo*c_tmp + s_tmp*cost
                _Bzt[3] = sintmpi*s_tmp
                lpoints += 1

            if t_aleft:
                _rAt[1] = radA / (1.0 + eccenA * math.cos(curr_longit - curr_step))
                _Axt[1] = math.cos(curr_longit - curr_step)
                _Ayt[1] = math.sin(curr_longit - curr_step)
                lpoints += 1

            if t_aright:
                _rAt[3] = radA / (1.0 + eccenA * math.cos(curr_longit + curr_step))
                _Axt[3] = math.cos(curr_longit + curr_step)
                _Ayt[3] = math.sin(curr_longit + curr_step)
                lpoints += 1
            
            if lpoints == 1:
                if t_aleft: i1max = 1
                if t_aright: i1min = 3
                if t_bleft: j1max = 1
                if t_bright: j1min = 3
            
            if lpoints == 2:
                if t_aleft and t_bright: calc1 = True
                if t_aleft and t_bleft: calc2 = True
                if t_aright and t_bright: calc3 = True
                if t_aright and t_bleft: calc4 = True

            j1_t, i1_t = 2, 2
            
            for j1 in range(j1min, j1max + 1):
                for i1 in range(i1min, i1max + 1):
                    if lpoints == 2:
                        if i1 != 1:
                            if ((j1 != 3) and calc1) or ((j1 != 1) and calc2): continue
                        if i1 != 3:
                            if ((j1 != 3) and calc3) or ((j1 != 1) and calc4): continue
                    
                    if i1 == 2 and j1 == 2: continue
                    
                    Dx = _rBt[j1]*_Bxt[j1] - _rAt[i1]*_Axt[i1]
                    Dy = _rBt[j1]*_Byt[j1] - _rAt[i1]*_Ayt[i1]
                    Dz = _rBt[j1]*_Bzt[j1]
                    dist_val = Dx*Dx + Dy*Dy + Dz*Dz
                    
                    if dist_val < curr_moid:
                        curr_moid = dist_val
                        j1_t = j1
                        i1_t = i1

            if j1_t != 2 or i1_t != 2:
                t_aleft, t_aright = False, False
                t_bleft, t_bright = False, False
                
                if i1_t != 2:
                    if i1_t == 1:
                        t_aleft = True
                        curr_longit -= curr_step
                        _rAt[3] = _rAt[2]; _Axt[3] = _Axt[2]; _Ayt[3] = _Ayt[2]
                        _rAt[2] = _rAt[1]; _Axt[2] = _Axt[1]; _Ayt[2] = _Ayt[1]
                    else:
                        t_aright = True
                        curr_longit += curr_step
                        _rAt[1] = _rAt[2]; _Axt[1] = _Axt[2]; _Ayt[1] = _Ayt[2]
                        _rAt[2] = _rAt[3]; _Axt[2] = _Axt[3]; _Ayt[2] = _Ayt[3]
                
                if j1_t != 2:
                    if j1_t == 1:
                        t_bleft = True
                        curr_trueB -= curr_step
                        _rBt[3] = _rBt[2]; _Bxt[3] = _Bxt[2]; _Byt[3] = _Byt[2]; _Bzt[3] = _Bzt[2]
                        _rBt[2] = _rBt[1]; _Bxt[2] = _Bxt[1]; _Byt[2] = _Byt[1]; _Bzt[2] = _Bzt[1]
                    else:
                        t_bright = True
                        curr_trueB += curr_step
                        _rBt[1] = _rBt[2]; _Bxt[1] = _Bxt[2]; _Byt[1] = _Byt[2]; _Bzt[1] = _Bzt[2]
                        _rBt[2] = _rBt[3]; _Bxt[2] = _Bxt[3]; _Byt[2] = _Byt[3]; _Bzt[2] = _Bzt[3]
            else:
                t_aleft, t_aright = True, True
                t_bleft, t_bright = True, True
                curr_step *= 0.15
        
        return curr_moid, curr_trueB, curr_longit

    # Run initial tuning for all candidates
    for jjj in range(nmax):
        res_moid, res_trueB, res_longit = run_tuning(
            tmpmoid[jjj], tmptrueB[jjj], tmplongit[jjj], stepini, steptresh
        )
        tmpmoid[jjj] = res_moid
        tmptrueB[jjj] = res_trueB
        tmplongit[jjj] = res_longit

    # --- 5. Selection and Final Result ---
    if nmax == 0:
        final_moid = dist_min
        # Fallback angle (approx)
        final_trueB = trueB_o
        final_longit = longit_o
    else:
        best_idx = 0
        best_val = tmpmoid[0]
        for i in range(1, nmax):
            if tmpmoid[i] < best_val:
                best_val = tmpmoid[i]
                best_idx = i
        
        # Final high precision tune on the best candidate
        final_moid, final_trueB, final_longit = run_tuning(
            tmpmoid[best_idx], 
            tmptrueB[best_idx], 
            tmplongit[best_idx], 
            2.0 * stepini, 
            stepmin
        )

    # --- 6. Formatting Output ---
    moid_au = math.sqrt(final_moid)
    moid_km = moid_au * AU_TO_KM
    
    # Convert radians to degrees and normalize to 0-360
    nu_A_deg = math.degrees(final_longit) % 360
    nu_B_deg = math.degrees(final_trueB) % 360

    return {
        "moid_au": moid_au,
        "moid_km": moid_km,
        "true_anomaly_A_deg": nu_A_deg,
        "true_anomaly_B_deg": nu_B_deg
    }
#Earth and Apophis
EA = wisric_moid((1.4765*10**8)*6.6846*10**-9, 9.166999*10**-3, math.radians(6.6437516*10), math.radians(1.476083*10), math.radians(4.242269*10**-3), 
            (1.379393*10**8)*6.6846*10**-9, 1.9097084*10**-1, math.radians(1.291994*10**2), math.radians(2.038196*10**2), math.radians(3.335653))
#Earth and 2024 YR4
EYR4 = wisric_moid((1.4765*10**8)*6.6846*10**-9, 9.166999*10**-3, math.radians(6.6437516*10), math.radians(1.476083*10), math.radians(4.242269*10**-3), 
            (3.7680703*10**8)*6.6846*10**-9, 6.616414*10**-1, math.radians(1.342990*10**2), math.radians(2.714790*10**2), math.radians(3.400149))
#Apophis and 2024 YR4
AYR4 = wisric_moid((1.379393*10**8)*6.6846*10**-9, 1.9097084*10**-1, math.radians(1.291994*10**2), math.radians(2.038196*10**2), math.radians(3.335653), 
            (3.7680703*10**8)*6.6846*10**-9, 6.616414*10**-1, math.radians(1.342990*10**2), math.radians(2.714790*10**2), math.radians(3.400149))
#Earth and 3I Atlas
E3I = wisric_moid((1.4765*10**8)*6.6846*10**-9, 9.166999*10**-3, math.radians(6.6437516*10), math.radians(1.476083*10), math.radians(4.242269*10**-3), 
            (-3.955266*10**7)*6.6846*10**-9, 6.146926, math.radians(1.281725*10**2), math.radians(3.2228906*10**2), math.radians(1.751250*10**2))
#Earth and Apophis with Sun centered elements from JPL
EAS = wisric_moid((1.494774301325314*10**8)*6.6846*10**-9, 1.591429536350342*10**-2, math.radians(2.863568704502947*10**2), math.radians(1.776179030721728*10**2), math.radians(3.549055731418342*10**-3), 
            (137986131.45738875866)*6.6846*10**-9, 0.1911663355386932, math.radians(126.6728325163065), math.radians(203.8996515621043), math.radians(3.340958441017069))
#Earth and 3I Atlas with Sun centered elements from JPL
E3IS = wisric_moid((1.494774301325314*10**8)*6.6846*10**-9, 1.591429536350342*10**-2, math.radians(2.863568704502947*10**2), math.radians(1.776179030721728*10**2), math.radians(3.549055731418342*10**-3), 
            (-3.948081413761597*10**7)*6.6846*10**-9, 6.139777539558833, math.radians(1.280059555748132*10**2), math.radians(3.221547051729693*10**2), math.radians(1.751134539606639*10**2))

In [17]:
E3IS

{'moid_au': 0.36590139189853077,
 'moid_km': 54738069.11418643,
 'true_anomaly_A_deg': 269.5695560001464,
 'true_anomaly_B_deg': 0.11866396801101496}